# Traditional Bayesian Analyses
This is tutorial for [MaCh3SBITools](https://github.com/henry-wallace-phys/MaCh3SbiTools) and aims to provide a practical guide to using MaCh3SBI tools.
It provides a fake physics engine with known results and will walk you through the entire analysis chain!


In [ ]:
# General Use Imports, run this before anything else!
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

from mach3sbitools.utils import MaCh3Logger

logger = MaCh3Logger(name="tutorial")

# Part 1: Traditional Bayesian Analyses
This section will walk you through working with a "traditional" Bayesian analysis with a mock physics "engine"

### The "Physics" Engine
The physics engine is designed to mimic the [MaCh3 Python Bindings](https://github.com/mach3-software/MaCh3).

It consists of two components:
* The [`ParameterHandler`](physics_engine/parameter_handler.py). This allows the user to set parameters to some value as well as containing information about the parameter priors
* The [`SampleHandler`](physics_engine/sample_handler.py). This handles the "simulation" step via the `SampleHandler.reweight()` method. It is designed to mimic the behaviour in a proper Monte Carlo reweighting loop.

It is configurable via [PhysicsConfig.yaml](physics_configs/PhysicsConfig.yaml).

Let's have a quick play in order to understand how it works!

In [ ]:
import yaml
from physics_engine import ParameterHandler, SampleHandler

For some simulators, the configuration may have different components that are read slightly differently, for example the physics config may contain a list of sub-configs that need to be passed to the sample config. We mimic this behaviour here.


In [ ]:
# First we get the config and check it exists
physics_config = Path("physics_configs/PhysicsConfig.yaml")

if not physics_config.is_file() and not physics_config.exists():
    raise FileNotFoundError("Config file not found.")

# We extract the sample config from the physics config. This is a tad hacky but
# is what's currently done in MaCh3
with open(physics_config) as f:
    physics_open = yaml.safe_load(f)
    sample_config = Path(physics_open["Sample"]["sample_config"])

# We also need to get the SAMPLE config from the physics config!
if not sample_config.is_file():
    raise ValueError("Config file not found.")

We can now actually start the physics engine up!

In [ ]:
# Now we can set up the physics engines.
parameter_handler = ParameterHandler(physics_config)
sample_handler = SampleHandler(sample_config, parameter_handler)

Now we have our physics engine set up, let's have a play!

In [ ]:
# Firstly let's plot our data
data = sample_handler.get_data()
data_bins = sample_handler.samples.bins
plt.stairs(data, data_bins, label="data")
plt.xlabel("Energy")
plt.show()

We can also set our parameters to some strange values and look at the simulation changing!

In [ ]:
# Okay now we have our nominal spectra we can have a play with our parameter values! Let's do a random throw
parameter_handler.set_parameter_values(np.array([2, 1, 0.1, 1]))
sample_handler.reweight()

# We then grab the MC values from the "simulator"
mc_vals = sample_handler.get_mc_vals()

plt.stairs(data, data_bins, label="data")
plt.stairs(mc_vals, data_bins, label="MC")
plt.xlabel("Energy")
plt.legend(loc="upper right")
plt.show()

We can even run a simple MCMC like analysis!

In [ ]:
from physics_engine import MCMCSampler

# We can run a super simple MCMC analysis on this
sampler = MCMCSampler(sample_handler, parameter_handler)

# We now sample the chain (this will take a couple of mins, get a coffee!)
mcmc_chain = sampler.sample(draws=100000, chains=4, cores=4)

Now let's look at the posteriors!

In [ ]:
import arviz as az

az.plot_pair(
    mcmc_chain,
    var_names=[
        parameter_handler.get_name(i) for i in range(parameter_handler.n_params)
    ],
    kind="kde",
    marginals=True,
    divergences=True,
)
plt.show()

Now we've done our plotting we can do the usual set of Bayesian inference! This has hopefully illustrated that these things can be quite slow to perform (imagine needing to repeat this process at lots of points with a physics emulator that's 100x slower!). So let's move onto SBI in (part 2)[<link>]